# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassan-tech-pro/ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

This is a transparent, non-production triage rule on the starter snapshot, not a validated model score. It follows the Week-4 eligible-page filters (`impressions_90d > 0`, `content_age_days >= 90`, one row per page), then assigns each page to the first matching rule:

1. `STALE_VISIBLE` → `HUMAN_REFRESH_REVIEW`: no recorded update for at least 180 days and at least 300 impressions in the trailing 90 days. Rank within this tier by visibility and staleness.
2. `PAGE_ONE_CTR_REVIEW` → `HUMAN_SNIPPET_REVIEW`: average position 1–10, at least 300 impressions, and CTR below the weighted page-one reference CTR. Rank within this tier by visible volume and CTR gap.
3. `STRIKING_DISTANCE_REVIEW` → `HUMAN_RELEVANCE_REVIEW`: average position >10 and <=20 with at least 300 impressions. Rank within this tier by visibility and proximity to page one.
4. Otherwise `MONITOR_ONLY`: do not put the page in the action queue.

The priority tier orders action types; the numeric score orders pages only within a tier. Neither score is a probability, a predicted lift, nor a causal estimate.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_RELATIVE_PATH = Path("content_refresh_anonymized.csv")
repo_candidates = [Path.cwd(), *Path.cwd().parents]
data_path = next(
    (root / DATA_RELATIVE_PATH for root in repo_candidates if (root / DATA_RELATIVE_PATH).is_file()),
    None,
)
DATA_URL = (
    "https://raw.githubusercontent.com/Hassan-tech-pro/ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)
if data_path is not None:
    raw_data = pd.read_csv(data_path)
    repo_root = data_path.resolve().parents[2]
else:
    raw_data = pd.read_csv(DATA_URL)
    repo_root = Path.cwd()

required_columns = {
    "content_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "cpc",
}
missing_columns = required_columns.difference(raw_data.columns)
if missing_columns:
    raise ValueError(f"Starter CSV is missing required columns: {sorted(missing_columns)}")

numeric_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "cpc",
]
for column in numeric_columns:
    raw_data[column] = pd.to_numeric(raw_data[column], errors="coerce")

pages = (
    raw_data.loc[
        raw_data["impressions_90d"].gt(0)
        & raw_data["content_age_days"].ge(90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

MIN_IMPRESSIONS_90D = 300
STALE_DAYS = 180
PAGE_ONE_MIN_IMPRESSIONS = 300
STRIKING_MIN_IMPRESSIONS = 300
EDITOR_REVIEW_MINUTES = 15

page_one_mask = (
    pages["avg_position"].gt(0)
    & pages["avg_position"].le(10)
    & pages["impressions_90d"].ge(PAGE_ONE_MIN_IMPRESSIONS)
    & pages["ctr"].notna()
)
page_one_rows = pages.loc[page_one_mask]
page_one_weighted_ctr = (
    100.0 * page_one_rows["clicks_90d"].sum() / page_one_rows["impressions_90d"].sum()
    if len(page_one_rows) and page_one_rows["impressions_90d"].sum() > 0
    else np.nan
)

stale_visible = (
    pages["days_since_last_update"].ge(STALE_DAYS)
    & pages["impressions_90d"].ge(MIN_IMPRESSIONS_90D)
)
page_one_ctr_review = (
    page_one_mask
    & pages["ctr"].lt(page_one_weighted_ctr)
    if pd.notna(page_one_weighted_ctr)
    else pd.Series(False, index=pages.index)
)
striking_distance = (
    pages["avg_position"].gt(10)
    & pages["avg_position"].le(20)
    & pages["impressions_90d"].ge(STRIKING_MIN_IMPRESSIONS)
)

pages["archetype"] = np.select(
    [stale_visible, page_one_ctr_review, striking_distance],
    ["MATURE_STALE_VISIBLE", "PAGE_ONE_LOW_CTR", "STRIKING_DISTANCE"],
    default="MONITOR_ONLY",
)
pages["reason_code"] = np.select(
    [stale_visible, page_one_ctr_review, striking_distance],
    ["STALE_VISIBLE", "PAGE_ONE_CTR_REVIEW", "STRIKING_DISTANCE_REVIEW"],
    default="MONITOR_ONLY",
)
pages["action_label"] = np.select(
    [stale_visible, page_one_ctr_review, striking_distance],
    ["HUMAN_REFRESH_REVIEW", "HUMAN_SNIPPET_REVIEW", "HUMAN_RELEVANCE_REVIEW"],
    default="MONITOR_ONLY",
)
pages["priority_tier"] = np.select(
    [stale_visible, page_one_ctr_review, striking_distance],
    [1, 2, 3],
    default=4,
).astype(int)

log_visibility = np.log1p(pages["impressions_90d"].clip(lower=0))
stale_factor = 1.0 + pages["days_since_last_update"].clip(lower=0, upper=365).fillna(0) / 365.0
ctr_gap_fraction = (
    (page_one_weighted_ctr - pages["ctr"]).clip(lower=0) / page_one_weighted_ctr
    if pd.notna(page_one_weighted_ctr) and page_one_weighted_ctr > 0
    else pd.Series(0.0, index=pages.index)
)
striking_factor = ((21.0 - pages["avg_position"]) / 10.0).clip(lower=0, upper=1)
pages["priority_score"] = np.select(
    [stale_visible, page_one_ctr_review, striking_distance],
    [
        log_visibility * stale_factor,
        log_visibility * ctr_gap_fraction,
        log_visibility * striking_factor,
    ],
    default=0.0,
)
pages["estimated_review_minutes"] = np.where(
    pages["priority_tier"].lt(4), EDITOR_REVIEW_MINUTES, 0
)
pages["click_cpc_value_proxy_90d"] = pages["clicks_90d"] * pages["cpc"]

queue = (
    pages.loc[pages["priority_tier"].lt(4)]
    .sort_values(
        ["priority_tier", "priority_score", "impressions_90d", "content_id"],
        ascending=[True, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

print(f"Source: {'local starter CSV' if data_path is not None else 'public starter CSV'}")
print(f"Eligible page rows: {len(pages):,}; prioritized review candidates: {len(queue):,}")
print(f"Page-one weighted CTR reference: {page_one_weighted_ctr:.3f}%")
print("Scores define within-tier triage order only; they are not model probabilities or expected lift.")

Source: public starter CSV
Eligible page rows: 30,000; prioritized review candidates: 10,521
Page-one weighted CTR reference: 0.359%
Scores define within-tier triage order only; they are not model probabilities or expected lift.


## 2. Intended use and limits

**Intended user:** an editor or SEO analyst who needs a short list of pages to inspect during a scheduled content review. The queue is triage support only; a person decides whether to refresh, improve a snippet, investigate relevance, or do nothing.

**What the evidence says:** the supplied March 2026 paper reports a lifecycle pattern (health highest at 61–90 days, lower at 271–365 days) and stronger reported metrics for some older pages refreshed recently. It also reports a stable 31–90-day freshness growth-to-decline ratio of 7.88:1, while warning that the 361+ bucket's 283:1 ratio comes from 283 growing pages and only one declining page. These are paper-reported observational comparisons; selection into refresh, page quality, and client/topic mix may confound them. They do not show that a refresh causes recovery.

**Scope limits:** this playbook uses one trailing-90-day starter snapshot, filtered to pages with impressions and age >=90 days. It has no page text, URL, query mix, editor effort, or post-action outcome. Its rule thresholds are provisional and not validated against future lift; do not treat the click × CPC proxy as revenue or expected uplift.

In [2]:
action_summary = (
    queue.groupby(["priority_tier", "archetype", "reason_code", "action_label"], dropna=False)
    .agg(
        page_count=("content_id", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        median_days_since_update=("days_since_last_update", "median"),
        pages_with_cpc=("cpc", "count"),
        click_cpc_value_proxy_sum=(
            "click_cpc_value_proxy_90d",
            lambda values: values.sum(min_count=1),
        ),
    )
    .reset_index()
    .sort_values("priority_tier")
)
print("Aggregate action counts and current-snapshot value context; no row identifiers shown.")
display(action_summary)

estimated_review_hours = queue["estimated_review_minutes"].sum() / 60.0
print(
    f"At {EDITOR_REVIEW_MINUTES} minutes per candidate, reviewing this queue would take "
    f"about {estimated_review_hours:.1f} editor hours. This is a planning assumption, not a measured cost."
)
print(
    "Click × CPC is a historical click-equivalent value proxy for rows with CPC data; "
    "it is not revenue, expected recovery, or incremental value from an action."
)

Aggregate action counts and current-snapshot value context; no row identifiers shown.


,priority_tier,archetype,reason_code,action_label,page_count,median_impressions_90d,median_days_since_update,pages_with_cpc,click_cpc_value_proxy_sum
0,1,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,22,1362.0,194.0,18,0.00
1,2,PAGE_ONE_LOW_CTR,PAGE_ONE_CTR_REVIEW,HUMAN_SNIPPET_REVIEW,5456,3604.0,22.0,5324,38379.79
2,3,STRIKING_DISTANCE,STRIKING_DISTANCE_REVIEW,HUMAN_RELEVANCE_REVIEW,5043,1783.0,22.0,4960,16478.54


At 15 minutes per candidate, reviewing this queue would take about 2630.2 editor hours. This is a planning assumption, not a measured cost.
Click × CPC is a historical click-equivalent value proxy for rows with CPC data; it is not revenue, expected recovery, or incremental value from an action.


## 3. Human review + the no-go list

A reviewer should verify the page in an authorized system before acting. This export contains pseudonymous IDs, not usable URLs or page copy.

**Required human checks:** confirm the correct page and owner; verify that the content is actually stale or that the snippet/query match is weak; inspect recent changes, cannibalization, seasonality, and current search demand; check brand, legal, and factual requirements; record the chosen action or a reason for rejecting the recommendation.

**Never automate from this queue:** publishing rewrites; deleting, merging, redirecting, or canonicalizing pages; changing titles at scale; promising traffic, ranking, or revenue gains; treating a proxy or the PDF's cohort comparisons as causal proof; overriding editorial, legal, or client decisions; inferring real identities from anonymized IDs.

In [3]:
human_review_rules = pd.DataFrame(
    [
        {"stage": "Identity", "review_rule": "Resolve the pseudonymous page ID only inside an authorized system; verify page, owner, and canonical destination."},
        {"stage": "Evidence", "review_rule": "Recheck impressions, position, CTR window, and recent edits; confirm the rule still matches the page's current state."},
        {"stage": "Content", "review_rule": "Read the page and assess factual freshness, intent match, quality, and whether an edit is actually needed."},
        {"stage": "Decision", "review_rule": "Choose refresh, snippet review, relevance review, monitor, or reject; record a human reason and owner."},
        {"stage": "Follow-up", "review_rule": "Compare later observed metrics with a defined baseline; do not attribute changes to the action without a stronger design."},
    ]
)
no_go_actions = pd.DataFrame(
    [
        {"never_automate": "Publish or rewrite content", "why": "The queue does not inspect page text, factual accuracy, brand voice, or legal requirements."},
        {"never_automate": "Delete, merge, redirect, or change canonical URLs", "why": "The snapshot lacks URL relationships, query overlap, and migration risk."},
        {"never_automate": "Promise ranking, traffic, or revenue lift", "why": "There is no causal estimate or validated future-outcome model."},
        {"never_automate": "Expose real identity from a pseudonymous ID", "why": "The release is anonymized and contains no safe identity mapping."},
    ]
)
print("Human review gates:")
display(human_review_rules)
print("No-go automation list:")
display(no_go_actions)

Human review gates:


,stage,review_rule
0,Identity,Resolve the pseudonymous page ID only inside a...
1,Evidence,"Recheck impressions, position, CTR window, and..."
2,Content,"Read the page and assess factual freshness, in..."
3,Decision,"Choose refresh, snippet review, relevance revi..."
4,Follow-up,Compare later observed metrics with a defined ...


No-go automation list:


,never_automate,why
0,Publish or rewrite content,"The queue does not inspect page text, factual ..."
1,"Delete, merge, redirect, or change canonical URLs","The snapshot lacks URL relationships, query ov..."
2,"Promise ranking, traffic, or revenue lift",There is no causal estimate or validated futur...
3,Expose real identity from a pseudonymous ID,The release is anonymized and contains no safe...


## 4. Monitoring / retrain triggers

This is a deterministic rule, so there is no model to retrain today. Recheck the thresholds and action mapping when the source schema, reporting window, tracking coverage, or editorial workflow changes. Compare candidate counts and aggregate metrics with the saved snapshot receipt; a substantial shift is a prompt for human investigation, not an automatic threshold change.

For any action a human chooses, review observed impressions, clicks, CTR, and position at 30/60/90 days. A future model is not eligible for retraining or deployment until it has a separately defined post-decision label, time-aligned features, client/time-aware validation, a baseline comparison, and acceptable failure review.

In [4]:
monitoring_triggers = pd.DataFrame(
    [
        {"trigger": "Source contract changes", "check": "Column, grain, filter, or reporting-window change", "response": "Pause exports; update the contract and recheck rule inputs."},
        {"trigger": "Coverage shift", "check": "Eligible-page count or missingness changes materially from the saved snapshot", "response": "Investigate tracking/source changes before comparing queue counts."},
        {"trigger": "Queue shift", "check": "Reason-code counts or page-one weighted CTR reference changes materially", "response": "Review bucket tables and thresholds with an editor; do not auto-adjust."},
        {"trigger": "After human action", "check": "Observe impressions, clicks, CTR, and position at 30/60/90 days", "response": "Record outcomes descriptively; use a suitable comparison design before causal language."},
        {"trigger": "Future model proposal", "check": "New future-window labels and grouped/time-aware holdout are available", "response": "Rebuild and validate against a frozen rule baseline before any use."},
    ]
)
current_queue_snapshot = pd.DataFrame(
    [
        {"metric": "eligible_unique_pages", "value": int(len(pages))},
        {"metric": "prioritized_review_candidates", "value": int(len(queue))},
        {"metric": "estimated_editor_review_hours_at_assumed_minutes", "value": round(float(estimated_review_hours), 2)},
        {"metric": "page_one_weighted_ctr_reference_pct", "value": round(float(page_one_weighted_ctr), 4) if pd.notna(page_one_weighted_ctr) else None},
    ]
)
print("Monitoring triggers:")
display(monitoring_triggers)
print("Aggregate reference values for the next snapshot; this run has no historical comparator:")
display(current_queue_snapshot)

Monitoring triggers:


,trigger,check,response
0,Source contract changes,"Column, grain, filter, or reporting-window change",Pause exports; update the contract and recheck...
1,Coverage shift,Eligible-page count or missingness changes mat...,Investigate tracking/source changes before com...
2,Queue shift,Reason-code counts or page-one weighted CTR re...,Review bucket tables and thresholds with an ed...
3,After human action,"Observe impressions, clicks, CTR, and position...",Record outcomes descriptively; use a suitable ...
4,Future model proposal,New future-window labels and grouped/time-awar...,Rebuild and validate against a frozen rule bas...


Aggregate reference values for the next snapshot; this run has no historical comparator:


,metric,value
0,eligible_unique_pages,30000.0000
1,prioritized_review_candidates,10521.0000
2,estimated_editor_review_hours_at_assumed_minutes,2630.2500
3,page_one_weighted_ctr_reference_pct,0.3594


## 5. Exports for the paper

On execution, write the ranked queue to `work/outputs/content_action_queue.csv` (generated data; keep it out of git), an aggregate metrics receipt to `work/outputs/action_playbook_metrics.json` (safe to commit), and a reason-code count figure to `work/figures/action_queue_by_reason.png` (safe to reuse in the paper). The notebook prints only aggregate totals and an ID-free queue preview.

In [5]:
import json

import matplotlib.pyplot as plt
from IPython.display import display

outputs_dir = repo_root / "work" / "outputs"
figures_dir = repo_root / "work" / "figures"
outputs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

queue_csv_path = outputs_dir / "content_action_queue.csv"
metrics_json_path = outputs_dir / "action_playbook_metrics.json"
figure_path = figures_dir / "action_queue_by_reason.png"

queue_columns = [
    "rank",
    "content_id",
    "archetype",
    "reason_code",
    "action_label",
    "priority_tier",
    "priority_score",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "cpc",
    "estimated_review_minutes",
    "click_cpc_value_proxy_90d",
]
queue[queue_columns].to_csv(queue_csv_path, index=False)

reason_counts = queue["reason_code"].value_counts().sort_index()
click_cpc_proxy_total = queue["click_cpc_value_proxy_90d"].sum(min_count=1)
metrics_receipt = {
    "dataset": "content_refresh_anonymized.csv",
    "lane": "Refresh / Content Opportunity Scoring",
    "selection_rule": {
        "starter_filters": ["impressions_90d > 0", "content_age_days >= 90", "deduplicated by content_id"],
        "stale_visible": "days_since_last_update >= 180 and impressions_90d >= 300",
        "page_one_ctr_review": "position 1-10, impressions_90d >= 300, CTR below page-one weighted reference",
        "striking_distance_review": "position > 10 and <= 20, impressions_90d >= 300",
    },
    "eligible_unique_pages": int(len(pages)),
    "prioritized_review_candidates": int(len(queue)),
    "queue_counts_by_reason_code": {str(key): int(value) for key, value in reason_counts.items()},
    "assumed_editor_review_minutes_per_candidate": int(EDITOR_REVIEW_MINUTES),
    "estimated_total_editor_review_hours": round(float(estimated_review_hours), 3),
    "page_one_weighted_ctr_reference_pct": (
        round(float(page_one_weighted_ctr), 6) if pd.notna(page_one_weighted_ctr) else None
    ),
    "queue_rows_with_cpc": int(queue["cpc"].notna().sum()),
    "sum_clicks_times_cpc_value_proxy_90d": (
        round(float(click_cpc_proxy_total), 6) if pd.notna(click_cpc_proxy_total) else None
    ),
    "interpretation": "Rule-based human-review triage; not a validated model, causal effect, revenue estimate, or expected uplift.",
    "source_rows_include_identifiers": False,
}
with metrics_json_path.open("w", encoding="utf-8") as metrics_file:
    json.dump(metrics_receipt, metrics_file, indent=2, allow_nan=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
if reason_counts.empty:
    ax.text(0.5, 0.5, "No pages met the playbook rules", ha="center", va="center")
    ax.set_axis_off()
else:
    ordered_counts = reason_counts.sort_values(ascending=True)
    ax.barh(ordered_counts.index, ordered_counts.values, color="#526c5b")
    ax.set_xlabel("Pages in human-review queue")
    ax.set_ylabel("Reason code")
    ax.set_title("Starter-snapshot review candidates by reason")
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.close(fig)

print(f"Queue CSV written: {queue_csv_path}")
print(f"Aggregate metrics JSON written: {metrics_json_path}")
print(f"Reusable aggregate figure written: {figure_path}")
print(f"Queue candidates: {len(queue):,}; estimated review time: {estimated_review_hours:.1f} hours")
print("Generated CSV stays out of git; aggregate JSON and figure are paper receipts.")
print("ID-free top-10 preview:")
display(
    queue.head(10)[
        ["rank", "archetype", "reason_code", "action_label", "priority_score", "impressions_90d", "ctr"]
    ]
)

Queue CSV written: /content/work/outputs/content_action_queue.csv
Aggregate metrics JSON written: /content/work/outputs/action_playbook_metrics.json
Reusable aggregate figure written: /content/work/figures/action_queue_by_reason.png
Queue candidates: 10,521; estimated review time: 2630.2 hours
Generated CSV stays out of git; aggregate JSON and figure are paper receipts.
ID-free top-10 preview:


,rank,archetype,reason_code,action_label,priority_score,impressions_90d,ctr
0,1,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,16.892059,61678,0.15
1,2,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,16.836280,59472,0.13
2,3,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,15.552251,25715,0.23
3,4,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,14.516438,13299,0.49
4,5,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,13.727729,7812,0.01
5,6,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,13.652646,7558,0.20
6,7,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,12.913441,4590,0.00
7,8,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,12.902057,4556,0.33
8,9,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,12.858769,4429,0.38
9,10,MATURE_STALE_VISIBLE,STALE_VISIBLE,HUMAN_REFRESH_REVIEW,12.520273,954,0.42


## Self-check

Do not mark these complete until the notebook is run and the generated files are inspected:

- [ ] Ranked queue has reason codes, action labels, and an archetype-to-action mapping
- [ ] Decay/refresh findings from the supplied paper are framed as observational
- [ ] Intended use, limits, human review, no-go automation, and monitoring triggers are stated
- [ ] Cost and click-CPC context are explicitly labeled as assumptions/proxies, not ROI or lift
- [ ] Queue CSV, aggregate metrics JSON, and aggregate figure are generated at the stated paths
- [ ] No client names, URLs, private queries, or row-level identifiers appear in notebook output
- [ ] Notebook runs top to bottom; I review the output and explain the recommendations
- [ ] Keep the generated queue CSV out of git; keep the aggregate JSON and reusable figure as paper receipts
- [ ] Work stays local in `work/notebooks/`; nothing is published to GitHub from this session